# Test notebook for `circle_results_tools.py`

This notebook is a lightweight template to test the external module with your current circle-subfield workflow.

You will need these available in your environment:

- `results_comp`
- `MAPS_DIR`
- `wcs`
- `name_file`
- `radius`
- `pc`

The module file should be placed where Python can import it, or you can append its folder to `sys.path`.

In [ ]:
from pathlib import Path
import sys

module_dir = Path("/mnt/data")
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

import circle_results_tools as crt

## 1. Define analysis inputs

Edit these values if needed.

In [ ]:
# Example placeholders: edit to your current values
name_file = "LVM-30Dor-6563"
catalog_path = "fits_ready/circle_catalog.csv"
filelist_path = "fits_ready/filelist_circles.txt"

# These should already exist in your notebook/session:
# MAPS_DIR
# results_comp
# wcs
# radius = 413 * u.arcsec
# pc = ...

## 2. Load the list of exported circle names

In [ ]:
with open(filelist_path, "r", encoding="utf-8") as f:
    names = [line.strip() for line in f if line.strip()]

names[:5], len(names)

## 3. Load all bundles

In [ ]:
bundles = {}
for name in names:
    bundles[name] = results_comp.load_line_bundle(name, name, MAPS_DIR, read_header=True)

dfs_obs  = {name: bundle["obs"]  for name, bundle in bundles.items()}
dfs_fit  = {name: bundle["fit"]  for name, bundle in bundles.items()}
meta_all = {name: bundle["meta"] for name, bundle in bundles.items()}

## 4. Load the circle catalog

In [ ]:
circles_df, circles = crt.load_circle_catalog(catalog_path, name_file)
circles_df.head()

## 5. Build the one-row-per-circle summary table

In [ ]:
results_df = crt.build_circle_results_df(
    circles,
    dfs_fit,
    radius=radius,
    pc=pc,
    param_row=0,
)

results_df.head()

## 6. Sky plot of fitted values inside the circles

In [ ]:
fig, ax = crt.plot_circle_fit_map(
    wcs,
    circles,
    dfs_fit,
    param="sig2",
    title="sig2 by circle",
    cmap="viridis",
    fontsize=8,
)

Try the same for `r0` and `m`.

In [ ]:
fig, ax = crt.plot_circle_fit_map(
    wcs,
    circles,
    dfs_fit,
    param="r0",
    title="r0 by circle",
    cmap="viridis",
    fontsize=8,
)

In [ ]:
fig, ax = crt.plot_circle_fit_map(
    wcs,
    circles,
    dfs_fit,
    param="m",
    title="m by circle",
    cmap="viridis",
    fontsize=8,
)

## 7. Ring-level statistics

In [ ]:
stats_sig_ring = crt.compute_group_stats(results_df, "sig", group_col="ring", order=["center", "inner", "middle", "outer"])
stats_sig_ring

In [ ]:
fig, ax, stats_df = crt.plot_parameter_by_ring(
    results_df,
    "sig",
    title="sig by ring",
    ylabel="sig",
)

## 8. Radius-in-pc version

In [ ]:
fig, ax, stats_df = crt.plot_parameter_by_radius(
    results_df,
    "sig",
    title="sig vs projected radius",
    ylabel="sig",
    jitter=0.02,
)

## 9. Repeat for `r0` and `m`

In [ ]:
fig, ax, stats_df = crt.plot_parameter_by_radius(
    results_df,
    "r0",
    title="r0 vs projected radius",
    ylabel="r0",
    jitter=0.02,
)

In [ ]:
fig, ax, stats_df = crt.plot_parameter_by_radius(
    results_df,
    "m",
    title="m vs projected radius",
    ylabel="m",
    jitter=0.02,
)